# All-in-One Image Restoration — Pilot Experiment v2 (Zero-Leakage & Controlled Branches)
**Mục tiêu**: Thực nghiệm đối chứng sòng phẳng giữa 4 nhánh (A, B, C, D) trên Google Colab Tesla T4 GPU.
- **Dữ liệu**: Khẳng định băm SHA-256 độc lập 100% giữa Train/Val và Test (Zero-Leakage Assert).
- **Đánh giá**: UnifiedEvaluator trên 4 tác vụ (Noise, Rain, Haze, Hybrid).
- **Đối chứng tham số**: Branch C có số tham số và layer giống hệt Branch B nhưng không tách tần số.
- **Khóa seed**: Chạy 2 seeds (42 và 123) cho mỗi nhánh, đo Mean ± Std và số giờ GPU.

## 1. Thiết lập Môi Trường & Kiểm Tra GPU Tesla T4

In [ ]:
# Kiểm tra GPU phần cứng
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Sẵn sàng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"Dung lượng VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Đồng Bộ Codebase Sang Đĩa SSD Cục Bộ Colab

In [ ]:
import os, shutil
from pathlib import Path

%cd /content
if not Path('/content/repo').exists():
    print("Đang clone codebase từ GitHub...")
    !git clone https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_repo
    shutil.move('/content/temp_repo', '/content/repo')

%cd /content/repo
!pip install -q -r requirements.txt
print(">> Môi trường thực thi Pilot v2 đã sẵn sàng!")

## 3. Chuẩn Bị Dữ Liệu v2 & Kiểm Tra Chống Rò Rỉ (Zero-Leakage Assertion)

In [ ]:
# Chuẩn bị 160 ảnh train sạch và 40 ảnh val sạch độc lập hoàn toàn với test
!python scripts/prepare_data_v2.py

## 4. Nhánh D: Đánh Giá Zero-Shot Baseline (PromptIR Pretrained)

In [ ]:
# Đo sàn baseline của PromptIR gốc chưa qua thích nghi
!python scripts/train_pilot_v2.py --branch D --seed 42

## 5. Nhánh A: Fine-tune PromptIR Trên Dữ Liệu v2 (Seed 42 & Seed 123)

In [ ]:
# Seed 42
!python scripts/train_pilot_v2.py --branch A --seed 42 --iters 1500 --batch_size 4

# Seed 123
!python scripts/train_pilot_v2.py --branch A --seed 123 --iters 1500 --batch_size 4

## 6. Nhánh B: Huấn Luyện FD-PromptIR Đề Xuất (Seed 42 & Seed 123)

In [ ]:
# Seed 42
!python scripts/train_pilot_v2.py --branch B --seed 42 --iters 1500 --batch_size 4

# Seed 123
!python scripts/train_pilot_v2.py --branch B --seed 123 --iters 1500 --batch_size 4

## 7. Nhánh C: Đối Chứng Tham Số Param-Matched PromptIR (Seed 42 & Seed 123)

In [ ]:
# Seed 42
!python scripts/train_pilot_v2.py --branch C --seed 42 --iters 1500 --batch_size 4

# Seed 123
!python scripts/train_pilot_v2.py --branch C --seed 123 --iters 1500 --batch_size 4

## 8. Tổng Hợp Kết Quả & Kiểm Định Tiêu Chí Thắng/Thua (Gate P4)

In [ ]:
import pandas as pd
import numpy as np

csv_path = 'results/pilot_v2_results.csv'
if Path(csv_path).exists():
    df = pd.read_csv(csv_path)
    print("=== BẢNG TỔNG HỢP KẾT QUẢ PILOT V2 ===")
    print(df.to_markdown(index=False))
    
    # Lọc các dòng best_val
    best_df = df[df['checkpoint_type'] == 'best_val']
    summary = best_df.groupby('branch')[['psnr_noise', 'psnr_rain', 'psnr_haze', 'psnr_hybrid', 'psnr_mean', 'gpu_hours']].agg(['mean', 'std'])
    print("\n=== THỐNG KÊ MEAN ± STD THEO NHÁNH (BEST VAL) ===")
    print(summary)
else:
    print("Chưa có file kết quả pilot_v2_results.csv")